# Notebook 04 — Semantic Harmonization

Loads the structural output and adds semantic annotation columns:
- `focus_raw` / `focus_normalized` — medical topic
- `primary_intent` — normalized intent taxonomy
- `dialogue_act` — turn-level dialogue act
- `medical_entities` — extracted medical entities (list)
- `annotation_source` — provenance of each annotation
- `annotation_confidence` — reliability score 0–1

All inference is **rule-based and deterministic**. No model calls are made unless explicitly noted.
Inferred labels are clearly flagged — they are not treated as ground truth.

In [1]:
import pandas as pd
import numpy as np
import re
import json
import warnings
from pathlib import Path
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')
tqdm.pandas()

STRUCTURAL_DIR = Path('../data/processed/structural')
SEMANTIC_DIR = Path('../data/processed/semantic')
SEMANTIC_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_parquet(STRUCTURAL_DIR / 'harmonized_structural.parquet')
print('Loaded structural dataset:', df.shape)
display(df.head(4))

Loaded structural dataset: (334033, 8)


,dialogue_id,turn_id,speaker,utterance,source_dataset,original_id,source_label_raw,dialogue_origin
0,medquad_000000,0,user,What is (are) Glaucoma ?,MedQuAD,0,Glaucoma,constructed
1,medquad_000000,1,assistant,Glaucoma is a group of diseases that can damag...,MedQuAD,0,Glaucoma,constructed
2,medquad_000001,0,user,What causes Glaucoma ?,MedQuAD,1,Glaucoma,constructed
3,medquad_000001,1,assistant,"Nearly 2.7 million people have glaucoma, a lea...",MedQuAD,1,Glaucoma,constructed


## 1. Medical Focus

### Strategy
1. **MedQuAD**: `source_label_raw` = `focus_area` → use directly (`annotation_source = source_metadata`)
2. **MedDialog**: No topic label in source. Infer from utterance using keyword matching.
3. **HealthChat**: `source_label_raw` contains JSON with `specialty` field → use specialty as focus.

`focus_normalized` applies a normalization dictionary to unify synonymous terms.

In [2]:
# ─── Normalization dictionary ────────────────────────────────────────────────
# Inspect and modify this dictionary to adjust normalization.
# Keys are lowercase strings to match against. Values are canonical forms.
FOCUS_NORMALIZATION = {
    # Diabetes
    'diabetes': 'diabetes',
    'diabetes mellitus': 'diabetes',
    'diabetic': 'diabetes',
    'diabetic condition': 'diabetes',
    'type 1 diabetes': 'diabetes_type1',
    'type 2 diabetes': 'diabetes_type2',
    'type i diabetes': 'diabetes_type1',
    'type ii diabetes': 'diabetes_type2',
    # Cardiac
    'heart attack': 'myocardial_infarction',
    'myocardial infarction': 'myocardial_infarction',
    'mi': 'myocardial_infarction',
    'heart disease': 'heart_disease',
    'cardiovascular disease': 'heart_disease',
    'coronary artery disease': 'coronary_artery_disease',
    'coronary heart disease': 'coronary_artery_disease',
    # Hypertension
    'hypertension': 'hypertension',
    'high blood pressure': 'hypertension',
    # Mental health
    'depression': 'depression',
    'major depressive disorder': 'depression',
    'mdd': 'depression',
    'anxiety': 'anxiety',
    'anxiety disorder': 'anxiety',
    'mental health': 'mental_health',
    # Cancer
    'cancer': 'cancer',
    'tumor': 'cancer',
    'tumour': 'cancer',
    'neoplasm': 'cancer',
    'breast cancer': 'breast_cancer',
    'lung cancer': 'lung_cancer',
    'prostate cancer': 'prostate_cancer',
    'colorectal cancer': 'colorectal_cancer',
    # Respiratory
    'asthma': 'asthma',
    'copd': 'copd',
    'chronic obstructive pulmonary disease': 'copd',
    # Neurological
    'migraine': 'migraine',
    'headache': 'headache',
    'alzheimer': 'alzheimers_disease',
    "alzheimer's disease": 'alzheimers_disease',
    'alzheimers disease': 'alzheimers_disease',
    'parkinson': 'parkinsons_disease',
    "parkinson's disease": 'parkinsons_disease',
    'parkinsons disease': 'parkinsons_disease',
    'stroke': 'stroke',
    # Musculoskeletal
    'arthritis': 'arthritis',
    'rheumatoid arthritis': 'rheumatoid_arthritis',
    'osteoarthritis': 'osteoarthritis',
    'back pain': 'back_pain',
    # Infections
    'covid': 'covid_19',
    'covid-19': 'covid_19',
    'coronavirus': 'covid_19',
    'influenza': 'influenza',
    'flu': 'influenza',
    'pneumonia': 'pneumonia',
    # Gastrointestinal
    'ibs': 'ibs',
    'irritable bowel syndrome': 'ibs',
    'gerd': 'gerd',
    'acid reflux': 'gerd',
    'gastroesophageal reflux': 'gerd',
    # Endocrine
    'thyroid': 'thyroid_disorder',
    'hypothyroidism': 'hypothyroidism',
    'hyperthyroidism': 'hyperthyroidism',
    # Eye
    'glaucoma': 'glaucoma',
    'macular degeneration': 'macular_degeneration',
}

def normalize_focus(focus_raw):
    """Apply normalization mapping to a raw focus string."""
    if focus_raw is None or (isinstance(focus_raw, float) and np.isnan(focus_raw)):
        return None
    key = str(focus_raw).lower().strip()
    if key in FOCUS_NORMALIZATION:
        return FOCUS_NORMALIZATION[key]
    # Try partial match for longer strings
    for k, v in FOCUS_NORMALIZATION.items():
        if k in key:
            return v
    # No match — clean and return as-is (lowercased, underscored)
    cleaned = re.sub(r'[^a-z0-9]+', '_', key).strip('_')
    return cleaned if cleaned else None

print('Normalization map size:', len(FOCUS_NORMALIZATION), 'entries')
# Quick test
tests = ['Diabetes Mellitus', 'Heart Attack', 'COPD', 'Glaucoma', 'Unknown Condition']
for t in tests:
    print(f'  {t!r:30s} -> {normalize_focus(t)!r}')

Normalization map size: 63 entries
  'Diabetes Mellitus'            -> 'diabetes'
  'Heart Attack'                 -> 'myocardial_infarction'
  'COPD'                         -> 'copd'
  'Glaucoma'                     -> 'glaucoma'
  'Unknown Condition'            -> 'unknown_condition'


In [3]:
# ─── Keyword-based focus inference for MedDialog (no source label) ───────────
# Ordered by specificity — more specific patterns first
FOCUS_KEYWORD_PATTERNS = [
    ('diabetes', re.compile(r'\bdiabet\w*\b', re.I)),
    ('hypertension', re.compile(r'\b(hypertension|high blood pressure)\b', re.I)),
    ('myocardial_infarction', re.compile(r'\b(heart attack|myocardial infarction|\bmi\b)\b', re.I)),
    ('heart_disease', re.compile(r'\b(heart disease|cardiovascular|cardiac)\b', re.I)),
    ('depression', re.compile(r'\b(depress\w+|major depressive)\b', re.I)),
    ('anxiety', re.compile(r'\banxiet\w+\b', re.I)),
    ('asthma', re.compile(r'\basthma\b', re.I)),
    ('copd', re.compile(r'\b(copd|chronic obstructive)\b', re.I)),
    ('cancer', re.compile(r'\b(cancer|tumor|tumour|neoplasm|carcinoma)\b', re.I)),
    ('back_pain', re.compile(r'\b(back pain|lumbar|spinal)\b', re.I)),
    ('migraine', re.compile(r'\b(migraine|migrain)\b', re.I)),
    ('headache', re.compile(r'\bheadache\b', re.I)),
    ('stroke', re.compile(r'\bstroke\b', re.I)),
    ('arthritis', re.compile(r'\barthritis\b', re.I)),
    ('covid_19', re.compile(r'\b(covid|coronavirus|sars-cov)\b', re.I)),
    ('influenza', re.compile(r'\b(influenza|\bflu\b)\b', re.I)),
    ('pneumonia', re.compile(r'\bpneumonia\b', re.I)),
    ('ibs', re.compile(r'\b(ibs|irritable bowel)\b', re.I)),
    ('gerd', re.compile(r'\b(gerd|acid reflux|heartburn|gastroesophageal)\b', re.I)),
    ('thyroid_disorder', re.compile(r'\bthyroid\b', re.I)),
    ('kidney_disease', re.compile(r'\b(kidney|renal)\b', re.I)),
    ('liver_disease', re.compile(r'\b(liver|hepatic|hepatitis)\b', re.I)),
    ('skin_condition', re.compile(r'\b(skin|rash|eczema|psoriasis|dermat\w+)\b', re.I)),
    ('infection', re.compile(r'\b(infection|bacterial|viral|antibiotic)\b', re.I)),
    ('vertigo', re.compile(r'\b(vertigo|dizziness|dizzy)\b', re.I)),
    ('pregnancy', re.compile(r'\b(pregnan|prenatal|maternal|obstetric)\b', re.I)),
    ('pain', re.compile(r'\b(pain|ache|sore|hurt\w*)\b', re.I)),
    ('fever', re.compile(r'\b(fever|temperature|pyrex\w*)\b', re.I)),
]

def infer_focus_from_text(text):
    """Infer focus topic from utterance text using keyword patterns."""
    if text is None or (isinstance(text, float) and np.isnan(text)):
        return None, 0.0
    for label, pattern in FOCUS_KEYWORD_PATTERNS:
        if pattern.search(str(text)):
            return label, 0.6  # Weak inference confidence
    return None, 0.0

# Test
test_texts = [
    'I have been feeling very dizzy and the room spins',
    'My blood sugar is high and I was diagnosed with diabetes',
    'I have severe chest pain after exercise',
]
for t in test_texts:
    label, conf = infer_focus_from_text(t)
    print(f'  {t[:60]!r} -> {label!r} (conf={conf})')

  'I have been feeling very dizzy and the room spins' -> 'vertigo' (conf=0.6)
  'My blood sugar is high and I was diagnosed with diabetes' -> 'diabetes' (conf=0.6)
  'I have severe chest pain after exercise' -> 'pain' (conf=0.6)


In [4]:
# ─── Apply focus annotation ───────────────────────────────────────────────────
focus_raw_list = []
focus_norm_list = []
focus_ann_src_list = []
focus_conf_list = []

for _, row in tqdm(df.iterrows(), total=len(df), desc='Focus annotation'):
    dataset = row['source_dataset']
    utterance = row['utterance']
    label_raw = row['source_label_raw']

    focus_raw = None
    focus_norm = None
    ann_src = 'rule_based'
    conf = 0.0

    if dataset == 'MedQuAD':
        # MedQuAD has focus_area directly in source_label_raw
        if label_raw and not (isinstance(label_raw, float) and np.isnan(label_raw)):
            focus_raw = str(label_raw)
            focus_norm = normalize_focus(focus_raw)
            ann_src = 'source_metadata'
            conf = 1.0
        else:
            # Fallback: infer from utterance
            focus_raw, conf = infer_focus_from_text(utterance)
            focus_norm = normalize_focus(focus_raw) if focus_raw else None
            ann_src = 'rule_based'

    elif dataset == 'MedDialog':
        # MedDialog has no topic label — infer from utterance
        focus_raw, conf = infer_focus_from_text(utterance)
        focus_norm = normalize_focus(focus_raw) if focus_raw else None
        ann_src = 'rule_based'

    elif dataset == 'HealthChat':
        # HealthChat has specialty classification in source_label_raw JSON
        try:
            meta = json.loads(str(label_raw))
            specialty = meta.get('specialty')
            if specialty and specialty != 'Other':
                focus_raw = specialty
                focus_norm = normalize_focus(specialty)
                ann_src = 'source_metadata'
                conf = 0.9  # Specialty classification, not free-text label
            else:
                focus_raw = specialty  # 'Other'
                focus_norm = 'general_medicine'
                ann_src = 'source_metadata'
                conf = 0.5
        except Exception:
            focus_raw = None
            focus_norm = None
            ann_src = 'rule_based'
            conf = 0.0

    focus_raw_list.append(focus_raw)
    focus_norm_list.append(focus_norm)
    focus_ann_src_list.append(ann_src)
    focus_conf_list.append(conf)

df['focus_raw'] = focus_raw_list
df['focus_normalized'] = focus_norm_list
df['_focus_ann_src'] = focus_ann_src_list
df['_focus_conf'] = focus_conf_list

print('Focus annotation complete.')
print('focus_raw non-null:', df['focus_raw'].notna().sum())
print('focus_normalized non-null:', df['focus_normalized'].notna().sum())
print()
print('Top focus_normalized values (MedQuAD):')
print(df[df['source_dataset']=='MedQuAD']['focus_normalized'].value_counts().head(10))

Focus annotation:   0%|          | 0/334033 [00:00<?, ?it/s]

Focus annotation complete.
focus_raw non-null: 257754
focus_normalized non-null: 257754

Top focus_normalized values (MedQuAD):
focus_normalized
myocardial_infarction    4072
cancer                   1166
diabetes                  568
parkinsons_disease        134
breast_cancer             116
hypertension              102
prostate_cancer            96
alzheimers_disease         96
stroke                     90
thyroid_disorder           86
Name: count, dtype: int64


## 2. Primary Intent

### Intent taxonomy
```
information_seeking, symptom_inquiry, diagnosis_inquiry, treatment_inquiry,
medication_inquiry, test_or_diagnosis, prevention, risk_factors, cause_or_mechanism,
prognosis, follow_up, clarification, emergency_or_urgent, other
```

### Strategy
1. **MedQuAD**: Question text contains strong lexical signals (`What is`, `What are the treatments`). Use pattern matching.
2. **MedDialog**: Patient input has symptom/treatment language. Pattern match on input.
3. **HealthChat**: No text — use taxonomy codes as proxy. Assistant turns = `other`.

In [5]:
VALID_INTENTS = {
    'information_seeking', 'symptom_inquiry', 'diagnosis_inquiry',
    'treatment_inquiry', 'medication_inquiry', 'test_or_diagnosis',
    'prevention', 'risk_factors', 'cause_or_mechanism', 'prognosis',
    'follow_up', 'clarification', 'emergency_or_urgent', 'other'
}

# ─── MedQuAD question patterns → intent mapping ────────────────────────────
# Applied in order — first match wins
INTENT_PATTERNS_USER = [
    ('treatment_inquiry',   re.compile(r'\b(treat|treatment|therapy|therapies|cure|manage|management|medication for|medicine for|drug for)\b', re.I)),
    ('medication_inquiry',  re.compile(r'\b(medication|medicine|drug|drugs|pill|prescription|dose|dosage|side effect)\b', re.I)),
    ('symptom_inquiry',     re.compile(r'\b(symptom|sign|feel|feeling|experience|suffer|complaint)\b', re.I)),
    ('diagnosis_inquiry',   re.compile(r'\b(diagnos|test|screening|detect|identify|how do (?:you|doctors?) know)\b', re.I)),
    ('test_or_diagnosis',   re.compile(r'\b(test|exam|lab|laboratory|imaging|scan|x-ray|mri|ct scan|biopsy)\b', re.I)),
    ('prevention',          re.compile(r'\b(prevent|prevention|avoid|reducing risk|lower risk|protective)\b', re.I)),
    ('risk_factors',        re.compile(r'\b(risk factor|risk of|who is at risk|prone to|predispos)\b', re.I)),
    ('cause_or_mechanism',  re.compile(r'\b(cause|causes|why|mechanism|what leads to|how does|origin|etiology)\b', re.I)),
    ('prognosis',           re.compile(r'\b(prognos|outcome|survival|life expectancy|recover|recovery|how long)\b', re.I)),
    ('emergency_or_urgent', re.compile(r'\b(emergency|urgent|immediately|chest pain|can.t breathe|severe|911|call doctor)\b', re.I)),
    ('information_seeking', re.compile(r'\b(what is|what are|tell me|explain|describe|definition|overview|information about)\b', re.I)),
]

# ─── HealthChat taxonomy code → intent mapping ─────────────────────────────
# A = Symptom/complaint categories
# B = Conversation type
# C = Specialty
# D = Context
TAXONOMY_INTENT_MAP = {
    'A1.1': 'symptom_inquiry',
    'A1.2': 'symptom_inquiry',
    'A1.3': 'diagnosis_inquiry',
    'A1.5': 'information_seeking',
    'A1.7': 'information_seeking',
    'B1':   'information_seeking',
    'B2':   'treatment_inquiry',
    'B3.3': 'medication_inquiry',
    'B4':   'prevention',
    'B5.1': 'cause_or_mechanism',
    'B5.2': 'risk_factors',
    'B7':   'prognosis',
    'B8':   'symptom_inquiry',
    'B8.1': 'symptom_inquiry',
    'B8.2': 'diagnosis_inquiry',
    'B9':   'information_seeking',
    'B10':  'treatment_inquiry',
    'C1':   'information_seeking',
    'C5':   'follow_up',
    'C6':   'emergency_or_urgent',
    'D1':   'information_seeking',
}

def infer_intent_from_text(text, speaker='user'):
    """Infer intent from utterance text."""
    if speaker == 'assistant':
        return 'other', 'rule_based', 0.9
    if text is None or (isinstance(text, float) and np.isnan(text)):
        return 'other', 'rule_based', 0.3
    text_str = str(text)
    for intent, pattern in INTENT_PATTERNS_USER:
        if pattern.search(text_str):
            return intent, 'rule_based', 0.7
    return 'information_seeking', 'rule_based', 0.5

def infer_intent_from_taxonomy(taxonomy_codes_json, speaker='user'):
    """Infer intent from HealthChat taxonomy codes."""
    if speaker == 'assistant':
        return 'other', 'derived_from_context', 0.9
    try:
        meta = json.loads(str(taxonomy_codes_json))
        codes = meta.get('taxonomy_codes', [])
        for code in codes:
            if code in TAXONOMY_INTENT_MAP:
                return TAXONOMY_INTENT_MAP[code], 'source_metadata', 0.8
    except Exception:
        pass
    return 'other', 'rule_based', 0.3

print('Intent pattern count:', len(INTENT_PATTERNS_USER))
print('Taxonomy intent map size:', len(TAXONOMY_INTENT_MAP))

# Test
test_qs = [
    ('What are the symptoms of diabetes?', 'user'),
    ('What treatments are available for glaucoma?', 'user'),
    ('How do I prevent heart disease?', 'user'),
    ('I have chest pain and difficulty breathing', 'user'),
]
for txt, spk in test_qs:
    intent, src, conf = infer_intent_from_text(txt, spk)
    print(f'  {txt[:60]!r} -> {intent!r} ({src}, {conf})')

Intent pattern count: 11
Taxonomy intent map size: 21
  'What are the symptoms of diabetes?' -> 'information_seeking' (rule_based, 0.7)
  'What treatments are available for glaucoma?' -> 'information_seeking' (rule_based, 0.5)
  'How do I prevent heart disease?' -> 'prevention' (rule_based, 0.7)
  'I have chest pain and difficulty breathing' -> 'emergency_or_urgent' (rule_based, 0.7)


In [6]:
# ─── Apply intent annotation ───────────────────────────────────────────────
intent_list = []
intent_ann_src_list = []
intent_conf_list = []

for _, row in tqdm(df.iterrows(), total=len(df), desc='Intent annotation'):
    dataset = row['source_dataset']
    utterance = row['utterance']
    speaker = row['speaker']
    label_raw = row['source_label_raw']

    if dataset in ('MedQuAD', 'MedDialog'):
        intent, ann_src, conf = infer_intent_from_text(utterance, speaker)
    elif dataset == 'HealthChat':
        intent, ann_src, conf = infer_intent_from_taxonomy(label_raw, speaker)
    else:
        intent, ann_src, conf = 'other', 'rule_based', 0.3

    intent_list.append(intent)
    intent_ann_src_list.append(ann_src)
    intent_conf_list.append(conf)

df['primary_intent'] = intent_list
df['_intent_ann_src'] = intent_ann_src_list
df['_intent_conf'] = intent_conf_list

print('Intent distribution:')
print(df['primary_intent'].value_counts())
print()
# Validate all values are in the allowed set
invalid = df[~df['primary_intent'].isin(VALID_INTENTS)]
print(f'Invalid intent values: {len(invalid)}')
assert len(invalid) == 0

Intent annotation:   0%|          | 0/334033 [00:00<?, ?it/s]

Intent distribution:
primary_intent
other                  158701
information_seeking     77051
symptom_inquiry         30304
treatment_inquiry       16245
medication_inquiry      15021
cause_or_mechanism       8809
diagnosis_inquiry        8370
risk_factors             5121
test_or_diagnosis        4991
emergency_or_urgent      4849
prognosis                2318
prevention               2124
follow_up                 129
Name: count, dtype: int64

Invalid intent values: 0


## 3. Dialogue Act

### Act taxonomy
```
question, answer, clarification_request, clarification, follow_up,
confirmation, correction, greeting, closing, statement, other
```

### Strategy
- Speaker + turn position (first/last) + punctuation + question words + linguistic patterns

In [7]:
VALID_DIALOGUE_ACTS = {
    'question', 'answer', 'clarification_request', 'clarification',
    'follow_up', 'confirmation', 'correction', 'greeting', 'closing',
    'statement', 'other'
}

# Precompile patterns
RE_QUESTION = re.compile(r'\?\s*$')
RE_QUESTION_WORD = re.compile(r'^\s*(what|how|why|when|where|who|which|can|could|should|is|are|do|does|did|has|have|will|would)\b', re.I)
RE_GREETING = re.compile(r'^\s*(hello|hi|hey|good morning|good afternoon|good evening|greetings|dear doctor|dear sir|dear madam)\b', re.I)
RE_CLOSING = re.compile(r'\b(thank you|thanks|goodbye|bye|regards|best regards|take care|god bless)\s*[.!]?\s*$', re.I)
RE_CLARIF_REQUEST = re.compile(r'\b(could you clarify|what do you mean|can you explain|please elaborate|i\'m not sure what|what exactly)\b', re.I)
RE_FOLLOW_UP = re.compile(r'^\s*(also|additionally|furthermore|another question|one more|follow.?up|what about|how about)\b', re.I)
RE_CONFIRMATION = re.compile(r'^\s*(yes|no|correct|right|exactly|confirmed|that.s right|that is right|indeed|absolutely|certainly)\b', re.I)

def infer_dialogue_act(speaker, utterance, turn_id, intent=None):
    """Infer dialogue act using rule-based approach."""
    if utterance is None or (isinstance(utterance, float) and np.isnan(utterance)):
        # HealthChat: no text available
        if speaker == 'user':
            return 'question', 'derived_from_context', 0.5
        else:
            return 'answer', 'derived_from_context', 0.5

    text = str(utterance).strip()

    # Greeting check — only if the turn is PREDOMINANTLY a greeting
    # (short text where greeting is the main content, not just a conversational opener)
    words = text.split()
    if RE_GREETING.match(text) and len(words) <= 10:
        return 'greeting', 'rule_based', 0.85

    # Closing check — only short turns where closing is the primary content
    if RE_CLOSING.search(text) and len(words) <= 15:
        return 'closing', 'rule_based', 0.85

    # Clarification request
    if RE_CLARIF_REQUEST.search(text):
        return 'clarification_request', 'rule_based', 0.8

    # Confirmation / correction
    if RE_CONFIRMATION.match(text) and len(text) < 100:
        return 'confirmation', 'rule_based', 0.75

    # Follow-up signals
    if RE_FOLLOW_UP.match(text):
        return 'follow_up', 'rule_based', 0.75

    if speaker == 'user':
        # Check if question
        if RE_QUESTION.search(text) or RE_QUESTION_WORD.match(text):
            # First turn and short = likely a question
            if intent == 'clarification':
                return 'clarification_request', 'rule_based', 0.75
            return 'question', 'rule_based', 0.85
        else:
            # User without question mark — likely a statement/description
            return 'statement', 'rule_based', 0.7
    else:  # assistant
        # Check if asking a clarifying question back
        if RE_QUESTION.search(text) and len(text) < 200:
            return 'clarification_request', 'rule_based', 0.75
        return 'answer', 'rule_based', 0.85

print('Testing dialogue act inference:')
tests = [
    ('user', 'What are the symptoms of glaucoma?', 0),
    ('user', 'I woke up feeling very dizzy this morning', 0),
    ('assistant', 'Glaucoma is a condition affecting the optic nerve...', 1),
    ('user', 'Thank you so much for your help!', 2),
    ('user', 'Hello, I have a question about my medication.', 0),
]
for spk, utt, tid in tests:
    act, src, conf = infer_dialogue_act(spk, utt, tid)
    print(f'  [{spk}] {utt[:60]!r} -> {act!r} (conf={conf})')

Testing dialogue act inference:
  [user] 'What are the symptoms of glaucoma?' -> 'question' (conf=0.85)
  [user] 'I woke up feeling very dizzy this morning' -> 'statement' (conf=0.7)
  [assistant] 'Glaucoma is a condition affecting the optic nerve...' -> 'answer' (conf=0.85)
  [user] 'Thank you so much for your help!' -> 'statement' (conf=0.7)
  [user] 'Hello, I have a question about my medication.' -> 'greeting' (conf=0.85)


In [8]:
# ─── Apply dialogue act annotation ────────────────────────────────────────
act_list = []
act_ann_src_list = []
act_conf_list = []

for _, row in tqdm(df.iterrows(), total=len(df), desc='Dialogue act'):
    act, ann_src, conf = infer_dialogue_act(
        row['speaker'], row['utterance'], row['turn_id'], row.get('primary_intent')
    )
    act_list.append(act)
    act_ann_src_list.append(ann_src)
    act_conf_list.append(conf)

df['dialogue_act'] = act_list
df['_act_ann_src'] = act_ann_src_list
df['_act_conf'] = act_conf_list

print('Dialogue act distribution:')
print(df['dialogue_act'].value_counts())

invalid = df[~df['dialogue_act'].isin(VALID_DIALOGUE_ACTS)]
print(f'Invalid dialogue act values: {len(invalid)}')
assert len(invalid) == 0

Dialogue act:   0%|          | 0/334033 [00:00<?, ?it/s]

Dialogue act distribution:
dialogue_act
answer                   154787
question                 103667
statement                 75120
clarification_request       440
greeting                     12
follow_up                     6
closing                       1
Name: count, dtype: int64
Invalid dialogue act values: 0


## 4. Medical Entity Extraction

Uses rule-based matching against a curated medical entity dictionary.
No NER model is called here — spaCy/scispaCy can be applied in a separate step if needed.

Entities are stored as a list (JSON-serializable). Empty list `[]` when none found.
HealthChat records (no text) always return `[]`.

In [9]:
# ─── Medical entity dictionary ─────────────────────────────────────────────
# Organized by category. Each entry: (canonical_entity, pattern)
MEDICAL_ENTITIES = [
    # diseases
    ('diabetes',              re.compile(r'\bdiabet\w*\b', re.I)),
    ('hypertension',          re.compile(r'\b(hypertension|high blood pressure)\b', re.I)),
    ('myocardial infarction', re.compile(r'\b(heart attack|myocardial infarction|\bmi\b)\b', re.I)),
    ('stroke',                re.compile(r'\bstroke\b', re.I)),
    ('asthma',                re.compile(r'\basthma\b', re.I)),
    ('copd',                  re.compile(r'\b(copd|chronic obstructive pulmonary)\b', re.I)),
    ('cancer',                re.compile(r'\b(cancer|carcinoma|neoplasm|tumor|tumour)\b', re.I)),
    ('glaucoma',              re.compile(r'\bglaucoma\b', re.I)),
    ('arthritis',             re.compile(r'\barthritis\b', re.I)),
    ('depression',            re.compile(r'\b(depression|depressive disorder)\b', re.I)),
    ('anxiety',               re.compile(r'\b(anxiety|anxiety disorder)\b', re.I)),
    ('pneumonia',             re.compile(r'\bpneumonia\b', re.I)),
    ('influenza',             re.compile(r'\b(influenza|\bflu\b)\b', re.I)),
    ('covid-19',              re.compile(r'\b(covid.?19|coronavirus|sars-cov-2)\b', re.I)),
    ('ibs',                   re.compile(r'\b(ibs|irritable bowel syndrome)\b', re.I)),
    ('gerd',                  re.compile(r'\b(gerd|acid reflux|gastroesophageal reflux)\b', re.I)),
    ('alzheimers disease',    re.compile(r"\b(alzheimer.?s disease|alzheimers|alzheimer)\b", re.I)),
    ('parkinsons disease',    re.compile(r"\b(parkinson.?s disease|parkinson)\b", re.I)),
    ('migraine',              re.compile(r'\bmigraine\b', re.I)),
    ('hypothyroidism',        re.compile(r'\bhypothyroidism\b', re.I)),
    ('hyperthyroidism',       re.compile(r'\bhyperthyroidism\b', re.I)),
    ('kidney disease',        re.compile(r'\b(kidney disease|renal disease|nephropathy)\b', re.I)),
    ('liver disease',         re.compile(r'\b(liver disease|hepatitis|cirrhosis)\b', re.I)),
    # symptoms
    ('dizziness',             re.compile(r'\b(dizziness|dizzy|vertigo)\b', re.I)),
    ('chest pain',            re.compile(r'\bchest pain\b', re.I)),
    ('nausea',                re.compile(r'\bnausea\b', re.I)),
    ('fever',                 re.compile(r'\bfever\b', re.I)),
    ('fatigue',               re.compile(r'\b(fatigue|tired|exhaustion)\b', re.I)),
    ('headache',              re.compile(r'\bheadache\b', re.I)),
    ('back pain',             re.compile(r'\bback pain\b', re.I)),
    ('shortness of breath',   re.compile(r'\b(shortness of breath|dyspnea|breathless)\b', re.I)),
    # medications
    ('insulin',               re.compile(r'\binsulin\b', re.I)),
    ('metformin',             re.compile(r'\bmetformin\b', re.I)),
    ('aspirin',               re.compile(r'\baspirin\b', re.I)),
    ('ibuprofen',             re.compile(r'\bibuprofen\b', re.I)),
    ('paracetamol',           re.compile(r'\b(paracetamol|acetaminophen)\b', re.I)),
    ('statins',               re.compile(r'\bstatin\w*\b', re.I)),
    ('antibiotics',           re.compile(r'\bantibiotic\w*\b', re.I)),
    ('beta blockers',         re.compile(r'\bbeta.?blocker\w*\b', re.I)),
    # body parts
    ('heart',                 re.compile(r'\bheart\b', re.I)),
    ('lung',                  re.compile(r'\b(lung|lungs|pulmonary)\b', re.I)),
    ('kidney',                re.compile(r'\b(kidney|kidneys|renal)\b', re.I)),
    ('liver',                 re.compile(r'\bliver\b', re.I)),
    ('brain',                 re.compile(r'\bbrain\b', re.I)),
    ('spine',                 re.compile(r'\b(spine|spinal|vertebra)\b', re.I)),
    # measurements
    ('blood glucose',         re.compile(r'\b(blood glucose|blood sugar|hba1c|a1c)\b', re.I)),
    ('blood pressure',        re.compile(r'\bblood pressure\b', re.I)),
    ('cholesterol',           re.compile(r'\bcholesterol\b', re.I)),
    # procedures
    ('surgery',               re.compile(r'\b(surgery|surgical|operation)\b', re.I)),
    ('biopsy',                re.compile(r'\bbiopsy\b', re.I)),
    ('mri',                   re.compile(r'\bmri\b', re.I)),
    ('ct scan',               re.compile(r'\bct scan\b', re.I)),
    ('x-ray',                 re.compile(r'\bx.?ray\b', re.I)),
]

print(f'Entity dictionary size: {len(MEDICAL_ENTITIES)} entries')

def extract_medical_entities(text):
    """Extract medical entities from text using dictionary matching.
    Returns a list of canonical entity strings (no duplicates).
    Returns [] if no entities found or text is unavailable.
    """
    if text is None or (isinstance(text, float) and np.isnan(text)):
        return []
    text_str = str(text)
    found = []
    seen = set()
    for entity, pattern in MEDICAL_ENTITIES:
        if entity not in seen and pattern.search(text_str):
            found.append(entity)
            seen.add(entity)
    return found

# Test
test_text = 'My blood glucose is very high and I was recently diagnosed with diabetes. My doctor prescribed metformin.'
print('Test extraction:', extract_medical_entities(test_text))

Entity dictionary size: 53 entries
Test extraction: ['diabetes', 'metformin', 'blood glucose']


In [10]:
# ─── Apply entity extraction ──────────────────────────────────────────────
df['medical_entities'] = df['utterance'].progress_apply(extract_medical_entities)

# Stats
has_entities = df['medical_entities'].apply(lambda x: len(x) > 0)
print(f'Turns with at least one entity: {has_entities.sum():,} / {len(df):,} ({100*has_entities.mean():.1f}%)')

# Most common entities
all_entities = [e for lst in df['medical_entities'] for e in lst]
print('\nTop 20 most common entities:')
print(pd.Series(all_entities).value_counts().head(20))

  0%|          | 0/334033 [00:00<?, ?it/s]

Turns with at least one entity: 141,032 / 334,033 (42.2%)

Top 20 most common entities:
surgery           18425
antibiotics       17339
heart             15732
cancer            14818
kidney            12674
fever             12580
x-ray             12166
liver             10932
lung              10625
mri                9905
spine              8459
anxiety            8194
brain              7895
blood pressure     7868
diabetes           7266
fatigue            5921
dizziness          5239
back pain          4943
hypertension       4852
biopsy             4804
Name: count, dtype: int64


## 5. Combine Annotation Source and Confidence

In [11]:
VALID_ANN_SOURCES = {'source_metadata', 'rule_based', 'derived_from_context', 'constructed', 'model_assisted'}

def combine_annotation_source(row):
    """
    Combine annotation sources from focus, intent, and act.
    Priority: source_metadata > derived_from_context > rule_based
    """
    sources = [row['_focus_ann_src'], row['_intent_ann_src'], row['_act_ann_src']]
    if 'source_metadata' in sources:
        return 'source_metadata'
    if 'derived_from_context' in sources:
        return 'derived_from_context'
    return 'rule_based'

def combine_annotation_confidence(row):
    """
    Combine confidences — use average of non-zero values.
    """
    confs = [row['_focus_conf'], row['_intent_conf'], row['_act_conf']]
    nonzero = [c for c in confs if c > 0]
    if not nonzero:
        return None
    return round(sum(nonzero) / len(nonzero), 3)

df['annotation_source'] = df.apply(combine_annotation_source, axis=1)
df['annotation_confidence'] = df.apply(combine_annotation_confidence, axis=1)

print('annotation_source distribution:')
print(df['annotation_source'].value_counts())
print()
print('annotation_confidence stats:')
print(df['annotation_confidence'].describe())

# Validate
invalid_src = df[~df['annotation_source'].isin(VALID_ANN_SOURCES)]
print(f'\nInvalid annotation_source: {len(invalid_src)}')
invalid_conf = df[df['annotation_confidence'].notna() & ((df['annotation_confidence'] < 0) | (df['annotation_confidence'] > 1))]
print(f'annotation_confidence out of [0,1]: {len(invalid_conf)}')
assert len(invalid_src) == 0
assert len(invalid_conf) == 0

annotation_source distribution:
annotation_source
rule_based         224358
source_metadata    109675
Name: count, dtype: int64

annotation_confidence stats:
count    334033.000000
mean          0.747406
std           0.088828
min           0.433000
25%           0.675000
50%           0.767000
75%           0.783000
max           0.917000
Name: annotation_confidence, dtype: float64

Invalid annotation_source: 0
annotation_confidence out of [0,1]: 0


## 6. Assemble Final Semantic DataFrame and Save

In [12]:
# Drop internal helper columns
DROP_COLS = ['_focus_ann_src', '_focus_conf', '_intent_ann_src', '_intent_conf', '_act_ann_src', '_act_conf']
semantic_df = df.drop(columns=DROP_COLS)

SEMANTIC_COLS = [
    'dialogue_id', 'turn_id', 'speaker', 'utterance',
    'focus_raw', 'focus_normalized', 'primary_intent', 'dialogue_act',
    'medical_entities',
    'source_dataset', 'original_id', 'source_label_raw', 'dialogue_origin',
    'annotation_source', 'annotation_confidence'
]
semantic_df = semantic_df[SEMANTIC_COLS]

print('Semantic dataset shape:', semantic_df.shape)
display(semantic_df.head(6))

Semantic dataset shape: (334033, 15)


,dialogue_id,turn_id,speaker,utterance,focus_raw,focus_normalized,primary_intent,dialogue_act,medical_entities,source_dataset,original_id,source_label_raw,dialogue_origin,annotation_source,annotation_confidence
0,medquad_000000,0,user,What is (are) Glaucoma ?,Glaucoma,glaucoma,information_seeking,question,[glaucoma],MedQuAD,0,Glaucoma,constructed,source_metadata,0.850
1,medquad_000000,1,assistant,Glaucoma is a group of diseases that can damag...,Glaucoma,glaucoma,other,answer,[glaucoma],MedQuAD,0,Glaucoma,constructed,source_metadata,0.917
2,medquad_000001,0,user,What causes Glaucoma ?,Glaucoma,glaucoma,cause_or_mechanism,question,[glaucoma],MedQuAD,1,Glaucoma,constructed,source_metadata,0.850
3,medquad_000001,1,assistant,"Nearly 2.7 million people have glaucoma, a lea...",Glaucoma,glaucoma,other,answer,"[glaucoma, blood pressure]",MedQuAD,1,Glaucoma,constructed,source_metadata,0.917
4,medquad_000002,0,user,What are the symptoms of Glaucoma ?,Glaucoma,glaucoma,information_seeking,question,[glaucoma],MedQuAD,2,Glaucoma,constructed,source_metadata,0.850
5,medquad_000002,1,assistant,Symptoms of Glaucoma Glaucoma can develop in ...,Glaucoma,glaucoma,other,answer,[glaucoma],MedQuAD,2,Glaucoma,constructed,source_metadata,0.917


In [13]:
out_path = SEMANTIC_DIR / 'harmonized_semantic.parquet'
semantic_df.to_parquet(out_path, index=False)
print(f'Saved: {out_path}')
print(f'Size: {out_path.stat().st_size / 1024 / 1024:.1f} MB')
print()
print('=== SEMANTIC HARMONIZATION SUMMARY ===')
print(f'Total turns: {len(semantic_df):,}')
print(f'Total dialogues: {semantic_df["dialogue_id"].nunique():,}')
print()
print('Intent distribution:')
print(semantic_df['primary_intent'].value_counts())
print()
print('Dialogue act distribution:')
print(semantic_df['dialogue_act'].value_counts())
print()
print('Annotation source distribution:')
print(semantic_df['annotation_source'].value_counts())
print()
print('Focus normalized (top 20):')
print(semantic_df['focus_normalized'].value_counts().head(20))

Saved: ..\data\processed\semantic\harmonized_semantic.parquet
Size: 84.2 MB

=== SEMANTIC HARMONIZATION SUMMARY ===
Total turns: 334,033
Total dialogues: 161,599

Intent distribution:
primary_intent
other                  158701
information_seeking     77051
symptom_inquiry         30304
treatment_inquiry       16245
medication_inquiry      15021
cause_or_mechanism       8809
diagnosis_inquiry        8370
risk_factors             5121
test_or_diagnosis        4991
emergency_or_urgent      4849
prognosis                2318
prevention               2124
follow_up                 129
Name: count, dtype: int64

Dialogue act distribution:
dialogue_act
answer                   154787
question                 103667
statement                 75120
clarification_request       440
greeting                     12
follow_up                     6
closing                       1
Name: count, dtype: int64

Annotation source distribution:
annotation_source
rule_based         224358
source_metadata  